In [5]:
import os, glob, math, random, json
import duckdb
import pandas as pd
import numpy as np

RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CONFIG = {
    "duckdb_path": "reddit_politics_v3.duckdb",
    "comments_glob": "politics/raw_comments/*.csv",
    "submissions_glob": "politics/raw_submissions/*.csv",
    "cache_dir": "cache_infer_v3",
    "outputs_dir": "outputs_infer_v3",

    "nela_db_path": "nela/nela-gt-2021.db",
    "nela_labels_csv": "nela/labels_2021.csv",

    "min_comments_thresholds": [18, 30, 50],
    "target_sample_n": 13350,

    "candidate_posts_topn": 2000,
    "holdout_sweep": [0.10, 0.20, 0.30, 0.40],
    "negatives_per_positive": 2,
    "dissimilar_quantile": 0.25,
    "max_users_for_holdout_eval": 3000,
}

os.makedirs(CONFIG["cache_dir"], exist_ok=True)
os.makedirs(CONFIG["outputs_dir"], exist_ok=True)

In [7]:
CSV_OPTS = """
    delim = ',',
    quote = '\"',
    escape = '\"',
    header = true,
    ignore_errors = true,
    all_varchar = true,
    null_padding = true,
    sample_size = 200,
    strict_mode = false
"""

def find_bad_files(path_glob: str) -> list[str]:
    bad = []
    for f in sorted(glob.glob(path_glob)):
        try:
            duckdb.sql(f"SELECT COUNT(*) FROM read_csv_auto('{f}', {CSV_OPTS});")
        except Exception:
            bad.append(f)
    return bad

bad_comments = find_bad_files(CONFIG["comments_glob"])
bad_subs = find_bad_files(CONFIG["submissions_glob"])

print("Bad comment files:", len(bad_comments))
print("Bad submission files:", len(bad_subs))


Bad comment files: 42
Bad submission files: 1


In [8]:
con = duckdb.connect(CONFIG["duckdb_path"])

comment_files = sorted(glob.glob(CONFIG["comments_glob"]))
sub_files = sorted(glob.glob(CONFIG["submissions_glob"]))

comment_files = [f for f in comment_files if f not in set(bad_comments)]
sub_files = [f for f in sub_files if f not in set(bad_subs)]

comment_list = "[" + ",".join(f"'{f}'" for f in comment_files) + "]"
sub_list = "[" + ",".join(f"'{f}'" for f in sub_files) + "]"

con.execute(f"""
CREATE OR REPLACE VIEW politics_comments AS
SELECT *
FROM read_csv_auto({comment_list},
    union_by_name=true,
    {CSV_OPTS}
);
""")

con.execute(f"""
CREATE OR REPLACE VIEW politics_submissions AS
SELECT *
FROM read_csv_auto({sub_list},
    union_by_name=true,
    {CSV_OPTS}
);
""")

summary = con.execute("""
SELECT 'comments' AS table, COUNT(*) AS n FROM politics_comments
UNION ALL
SELECT 'submissions', COUNT(*) FROM politics_submissions;
""").df()
summary

,table,n
0,comments,1293380
1,submissions,206184


In [10]:
def get_cols(con, view_name: str) -> set[str]:
    df = con.execute(f"DESCRIBE {view_name}").df()
    return set(df["column_name"].astype(str).tolist())

comment_cols = get_cols(con, "politics_comments")
sub_cols = get_cols(con, "politics_submissions")

print("comments cols (sample):", sorted(list(comment_cols))[:30])
print("submissions cols (sample):", sorted(list(sub_cols))[:30])

comments cols (sample): ['all_awardings', 'approved_at_utc', 'associated_award', 'author', 'author_cakeday', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_id', 'author_is_blocked', 'author_patreon_flair', 'author_premium', 'awarders', 'banned_at_utc', 'body', 'can_gild', 'can_mod_post', 'collapsed', 'collapsed_because_crowd_control', 'collapsed_reason', 'collapsed_reason_code', 'column00', 'comment_type', 'created', 'created_utc']
submissions cols (sample): ['all_awardings', 'allow_live_comments', 'approved_at_utc', 'archived', 'author', 'author_cakeday', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_id', 'author_is_blocked', 'author_patreon_flai

In [11]:
comment_cols = get_cols(con, "politics_comments")
sub_cols = get_cols(con, "politics_submissions")

comment_fullname_expr = None
if "name" in comment_cols:
    comment_fullname_expr = "lower(trim(name))"
elif "id" in comment_cols:
    comment_fullname_expr = "lower('t1_' || trim(id))"
elif "permalink" in comment_cols:
    comment_fullname_expr = "lower(trim(permalink))"
else:
    comment_fullname_expr = "NULL"

link_id_expr = None
if "link_id" in comment_cols:
    link_id_expr = "lower(trim(link_id))"
elif "link_permalink" in comment_cols:
    link_id_expr = "lower(trim(link_permalink))"
else:
    link_id_expr = "NULL"

parent_id_expr = "lower(trim(parent_id))" if "parent_id" in comment_cols else "NULL"

author_expr = "lower(trim(author))" if "author" in comment_cols else "NULL"
body_expr = "body" if "body" in comment_cols else "''"
created_expr = "created_utc" if "created_utc" in comment_cols else "NULL"
score_expr = "score" if "score" in comment_cols else "NULL"

submission_fullname_expr = None
if "name" in sub_cols:
    submission_fullname_expr = "lower(trim(name))"
elif "id" in sub_cols:
    submission_fullname_expr = "lower('t3_' || trim(id))"
else:
    submission_fullname_expr = "NULL"

submission_id_expr = "lower(trim(id))" if "id" in sub_cols else "NULL"
subreddit_expr = "lower(trim(subreddit))" if "subreddit" in sub_cols else ("lower(trim(subreddit_name_prefixed))" if "subreddit_name_prefixed" in sub_cols else "''")
title_expr = "title" if "title" in sub_cols else "''"
selftext_expr = "COALESCE(selftext, '')" if "selftext" in sub_cols else "''"
sub_created_expr = "created_utc" if "created_utc" in sub_cols else "NULL"
sub_score_expr = "score" if "score" in sub_cols else "NULL"
num_comments_expr = "num_comments" if "num_comments" in sub_cols else ("comment_count" if "comment_count" in sub_cols else "NULL")

sql = f"""
CREATE OR REPLACE TABLE comments_clean AS
SELECT
    {author_expr} AS author,
    {comment_fullname_expr} AS comment_fullname,
    {parent_id_expr} AS parent_id,
    {link_id_expr} AS link_id,
    {body_expr} AS body,
    TRY_CAST({created_expr} AS BIGINT) AS created_utc,
    TRY_CAST({score_expr} AS DOUBLE) AS score
FROM politics_comments
WHERE author IS NOT NULL
  AND lower(trim(author)) NOT IN ('[deleted]','automoderator','[removed]')
  AND {body_expr} IS NOT NULL
  AND regexp_matches({created_expr}, '^[0-9]+$')
;

CREATE OR REPLACE TABLE submissions_clean AS
SELECT
    {submission_fullname_expr} AS submission_fullname,
    {submission_id_expr} AS submission_id,
    {subreddit_expr} AS subreddit,
    {title_expr} AS title,
    {selftext_expr} AS selftext,
    TRY_CAST({sub_created_expr} AS BIGINT) AS created_utc,
    TRY_CAST({sub_score_expr} AS DOUBLE) AS score,
    TRY_CAST({num_comments_expr} AS BIGINT) AS num_comments
FROM politics_submissions
WHERE {submission_id_expr} IS NOT NULL
  AND regexp_matches({sub_created_expr}, '^[0-9]+$')
;
"""
con.execute(sql)

con.execute("CREATE INDEX IF NOT EXISTS idx_comments_author ON comments_clean(author);")
con.execute("CREATE INDEX IF NOT EXISTS idx_comments_link ON comments_clean(link_id);")
con.execute("CREATE INDEX IF NOT EXISTS idx_sub_id ON submissions_clean(submission_id);")

con.execute("""
SELECT
  (SELECT COUNT(*) FROM comments_clean) AS n_comments,
  (SELECT COUNT(*) FROM submissions_clean) AS n_submissions
""").df()


,n_comments,n_submissions
0,1234826,206175


In [12]:
con.execute("""
ALTER TABLE comments_clean DROP COLUMN IF EXISTS link_submission_id;
""")

con.execute("""
ALTER TABLE comments_clean ADD COLUMN link_submission_id VARCHAR;
""")

con.execute("""
UPDATE comments_clean
SET link_submission_id =
  CASE
    WHEN link_id IS NULL THEN NULL
    WHEN regexp_matches(link_id, '^t3_[a-z0-9]+$') THEN replace(link_id, 't3_', '')
    WHEN regexp_matches(link_id, '^[a-z0-9]{3,10}$') THEN link_id
    WHEN regexp_matches(link_id, '/comments/[a-z0-9]{3,10}/') THEN regexp_extract(link_id, '/comments/([a-z0-9]{3,10})/', 1)
    ELSE NULL
  END;
""")

con.execute("CREATE INDEX IF NOT EXISTS idx_comments_link_subid ON comments_clean(link_submission_id);")

con.execute("""
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN link_submission_id IS NOT NULL THEN 1 ELSE 0 END) AS with_subid
FROM comments_clean
""").df()


,n,with_subid
0,1234826,1234792.0


In [13]:
profiles = con.execute("""
WITH base AS (
  SELECT
    author,
    created_utc,
    score,
    length(body) AS body_len
  FROM comments_clean
  WHERE created_utc IS NOT NULL
),
per_author AS (
  SELECT
    author,
    COUNT(*) AS n_comments,
    AVG(score) AS avg_score,
    SUM(score) AS sum_score,
    AVG(body_len) AS avg_length,
    STDDEV_SAMP(body_len) AS std_length,
    MIN(created_utc) AS min_utc,
    MAX(created_utc) AS max_utc
  FROM base
  GROUP BY author
),
with_days AS (
  SELECT
    *,
    GREATEST(1.0, (max_utc - min_utc) / 86400.0) AS active_days
  FROM per_author
)
SELECT
  author,
  n_comments,
  avg_score,
  sum_score,
  avg_length,
  std_length,
  active_days,
  (n_comments / active_days) AS comments_per_day
FROM with_days
""").df()

profiles.head()


,author,n_comments,avg_score,sum_score,avg_length,std_length,active_days,comments_per_day
0,nothing_clever,17,1.0,8.0,228.705882,186.468082,460.381481,0.036926
1,420cherubi,16,1.0,16.0,112.437500,95.430931,690.327326,0.023177
2,yaddablahetc,7,1.0,5.0,53.857143,44.167969,316.113090,0.022144
3,rdpirate,1,1.0,1.0,157.000000,NaN,1.000000,1.000000
4,funnybunnyofdoom,4,1.0,4.0,54.000000,38.000000,49.594340,0.080654


In [14]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

def add_sentiment_features(con, profiles: pd.DataFrame, max_comments_per_user: int = 50) -> pd.DataFrame:
    authors = profiles["author"].tolist()
    if not authors:
        return profiles

    author_list = "(" + ",".join("'" + a.replace("'", "''") + "'" for a in authors) + ")"

    df = con.execute(f"""
    WITH sampled AS (
      SELECT
        author,
        body,
        ROW_NUMBER() OVER (PARTITION BY author ORDER BY created_utc DESC) AS rn
      FROM comments_clean
      WHERE author IN {author_list}
    )
    SELECT author, body
    FROM sampled
    WHERE rn <= {int(max_comments_per_user)}
    """).df()

    df["body"] = df["body"].fillna("").astype(str)
    df["compound"] = df["body"].map(lambda t: sid.polarity_scores(t)["compound"])
    per = df.groupby("author")["compound"].agg(["mean", "std"]).reset_index()
    per.columns = ["author", "avg_sentiment", "std_sentiment"]

    out = profiles.merge(per, on="author", how="left")
    out["avg_sentiment"] = out["avg_sentiment"].fillna(0.0)
    out["std_sentiment"] = out["std_sentiment"].fillna(0.0)
    return out

profiles = add_sentiment_features(con, profiles, max_comments_per_user=50)
profiles[["avg_sentiment","std_sentiment"]].describe()


,avg_sentiment,std_sentiment
count,258706.000000,258706.000000
mean,-0.004492,0.231064
std,0.391845,0.271969
min,-1.000000,0.000000
25%,-0.225550,0.000000
50%,0.000000,0.036487
75%,0.226300,0.461528
max,0.999300,1.386141


In [15]:
eps = 1e-6
profiles["sentiment_extremity"] = profiles["avg_sentiment"].abs()
profiles["rigidity"] = (profiles["sentiment_extremity"] / (profiles["std_sentiment"] + eps)).clip(0, 10.0)

profiles[["rigidity","sentiment_extremity","comments_per_day"]].describe()

,rigidity,sentiment_extremity,comments_per_day
count,258706.000000,258706.000000,258706.000000
mean,3.987400,0.290601,0.949262
std,4.617904,0.262894,1.430295
min,0.000000,0.000000,0.000170
25%,0.142749,0.062400,0.038315
50%,0.707105,0.226100,1.000000
75%,10.000000,0.458800,1.000000
max,10.000000,1.000000,71.000000


In [16]:
def add_influence_proxy(con, profiles: pd.DataFrame) -> pd.DataFrame:
    authors = profiles["author"].tolist()
    if not authors:
        return profiles

    author_list = "(" + ",".join("'" + a.replace("'", "''") + "'" for a in authors) + ")"

    replies_received = con.execute(f"""
    WITH parents AS (
      SELECT
        author AS replier,
        parent_id AS parent_fullname
      FROM comments_clean
      WHERE author IN {author_list}
        AND parent_id IS NOT NULL
        AND parent_id LIKE 't1_%'
    ),
    parent_authors AS (
      SELECT
        p.parent_fullname,
        pc.author AS parent_author
      FROM parents p
      JOIN comments_clean pc
        ON pc.comment_fullname = p.parent_fullname
      WHERE pc.author IS NOT NULL
    )
    SELECT parent_author AS author, COUNT(*) AS replies_received
    FROM parent_authors
    GROUP BY parent_author
    """).df()

    out = profiles.merge(replies_received, on="author", how="left")
    out["replies_received"] = out["replies_received"].fillna(0).astype(int)
    out["influence_basic"] = (
        np.log1p(out["sum_score"].clip(lower=0)) +
        0.5 * np.log1p(out["replies_received"])
    )
    return out

profiles = add_influence_proxy(con, profiles)
profiles[["influence_basic","replies_received"]].describe()


,influence_basic,replies_received
count,218247.000000,258706.000000
mean,1.704039,3.400605
std,1.257997,10.377524
min,0.000000,0.000000
25%,0.693147,0.000000
50%,1.242453,1.000000
75%,2.282174,3.000000
max,12.260935,689.000000


In [17]:
from scipy.stats import ks_2samp

def jsd(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / (p.sum() + eps)
    q = q / (q.sum() + eps)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log((p + eps) / (m + eps)))
    kl_qm = np.sum(q * np.log((q + eps) / (m + eps)))
    return 0.5 * (kl_pm + kl_qm)

def hist_prob(x, bins):
    h, _ = np.histogram(x, bins=bins)
    return h.astype(float)

def divergence_report(full: pd.DataFrame, samp: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        xf = full[col].replace([np.inf,-np.inf], np.nan).dropna().values
        xs = samp[col].replace([np.inf,-np.inf], np.nan).dropna().values
        if len(xf) < 10 or len(xs) < 10:
            continue
        ks = ks_2samp(xf, xs).statistic
        bins = np.quantile(xf, np.linspace(0,1,21))
        bins = np.unique(bins)
        if len(bins) < 5:
            bins = 20
        p = hist_prob(xf, bins)
        q = hist_prob(xs, bins)
        rows.append({"col": col, "ks": float(ks), "jsd": float(jsd(p,q))})
    return pd.DataFrame(rows).sort_values(["jsd","ks"], ascending=False)

def sample_threshold(df, min_comments, n=None):
    out = df[df["n_comments"] >= min_comments].copy()
    if n is not None and len(out) > n:
        out = out.sample(n=n, random_state=RANDOM_SEED)
    return out

def sample_stratified(df, n, activity_bins=10, rigidity_bins=3):
    x = df.copy()
    x["act_bin"] = pd.qcut(x["comments_per_day"].rank(method="first"), q=activity_bins, labels=False, duplicates="drop")
    x["rig_bin"] = pd.qcut(x["rigidity"].rank(method="first"), q=rigidity_bins, labels=False, duplicates="drop")
    grp = x.groupby(["act_bin","rig_bin"], dropna=False)
    per = max(1, n // max(1, grp.ngroups))
    parts = []
    for _, g in grp:
        take = min(len(g), per)
        parts.append(g.sample(n=take, random_state=RANDOM_SEED))
    out = pd.concat(parts, axis=0)
    if len(out) > n:
        out = out.sample(n=n, random_state=RANDOM_SEED)
    return out.drop(columns=["act_bin","rig_bin"])

def sample_distmatch(df, n, cols=("comments_per_day","rigidity")):
    x = df.copy()
    w = np.ones(len(x), dtype=float)
    for col in cols:
        q = np.quantile(x[col].values, np.linspace(0,1,11))
        q = np.unique(q)
        if len(q) < 4:
            continue
        bins = np.digitize(x[col].values, q[1:-1], right=True)
        counts = np.bincount(bins, minlength=bins.max()+1).astype(float)
        target = counts / counts.sum()
        inv = 1.0 / np.maximum(target[bins], 1e-6)
        w *= inv
    w = w / w.sum()
    idx = np.random.choice(np.arange(len(x)), size=min(n, len(x)), replace=False, p=w)
    return x.iloc[idx].copy()

def sample_influence_weighted(df, n, influence_col="influence_basic", temperature=1.0):
    x = df.copy()
    s = x[influence_col].astype(float).values
    s = (s - np.nanmin(s)) / (np.nanmax(s) - np.nanmin(s) + 1e-12)
    logits = s / max(temperature, 1e-6)
    w = np.exp(logits - np.max(logits))
    w = w / w.sum()
    idx = np.random.choice(np.arange(len(x)), size=min(n, len(x)), replace=False, p=w)
    return x.iloc[idx].copy()

FULL = profiles.copy()
FULL = FULL.replace([np.inf,-np.inf], np.nan).fillna(0)

KEY_COLS = ["n_comments","comments_per_day","avg_length","avg_sentiment","rigidity","influence_basic"]

samples = {}
for thr in CONFIG["min_comments_thresholds"]:
    samples[f"threshold_{thr}"] = sample_threshold(FULL, thr, n=CONFIG["target_sample_n"])

samples["stratified"] = sample_stratified(FULL, CONFIG["target_sample_n"])
samples["distmatch"] = sample_distmatch(FULL, CONFIG["target_sample_n"])
samples["influence_weighted"] = sample_influence_weighted(FULL, CONFIG["target_sample_n"], temperature=1.0)

reports = []
for name, samp in samples.items():
    rep = divergence_report(FULL, samp, KEY_COLS)
    rep["sample"] = name
    reports.append(rep)

div_df = pd.concat(reports, axis=0).reset_index(drop=True)
div_df.head(20)


,col,ks,jsd,sample
0,n_comments,0.948397,0.590211,threshold_18
1,influence_basic,0.800172,0.423330,threshold_18
2,comments_per_day,0.576997,0.324198,threshold_18
3,rigidity,0.524309,0.261446,threshold_18
4,avg_sentiment,0.224258,0.174114,threshold_18
5,avg_length,0.303279,0.091673,threshold_18
6,n_comments,0.975733,0.590211,threshold_30
7,influence_basic,0.858595,0.484632,threshold_30
8,comments_per_day,0.584850,0.371984,threshold_30
9,rigidity,0.552553,0.280736,threshold_30


In [18]:
candidate_posts = con.execute(f"""
SELECT
  submission_id,
  title,
  selftext,
  created_utc,
  COALESCE(num_comments, 0) AS num_comments
FROM submissions_clean
WHERE submission_id IS NOT NULL
ORDER BY COALESCE(num_comments, 0) DESC
LIMIT {int(CONFIG["candidate_posts_topn"])}
""").df()

candidate_posts["submission_id"] = candidate_posts["submission_id"].astype(str)
candidate_posts["text"] = (candidate_posts["title"].fillna("") + " " + candidate_posts["selftext"].fillna("")).str.strip()
candidate_posts = candidate_posts[candidate_posts["text"].str.len() > 0].reset_index(drop=True)

candidate_posts.head()


,submission_id,title,selftext,created_utc,num_comments,text
0,jpdbzl,Discussion Thread: 2020 General Election Part ...,Good afternoon r/Politics! Results can be foun...,1604696378,61437,Discussion Thread: 2020 General Election Part ...
1,ecm1zg,Megathread: House Votes to Impeach President D...,The United States House of Representatives vot...,1576719373,56322,Megathread: House Votes to Impeach President D...
2,9uvve1,Discussion Megathread: US Midterm Elections 20...,#Midterms 2018!\n\nToday is the day you’ve all...,1541562628,37064,Discussion Megathread: US Midterm Elections 20...
3,4v0hyx,2016 Democratic National Convention - Final Day,"Hello!\n\nToday is the fourth, and final day o...",1469709729,36223,2016 Democratic National Convention - Final Da...
4,4uon3q,2016 Democratic National Convention - Day 2,Hello!\n\nYesterday was very busy for us! Adm...,1469540199,32927,2016 Democratic National Convention - Day 2 He...


In [19]:
import spacy

nlp = spacy.load("en_core_web_md")

def spacy_embed_texts(texts, batch_size=128):
    vecs = []
    for doc in nlp.pipe(texts, batch_size=batch_size, n_process=1):
        vecs.append(doc.vector)
    return np.vstack(vecs)

post_vec_path = os.path.join(CONFIG["cache_dir"], "candidate_post_vecs.npy")
if os.path.exists(post_vec_path):
    post_vecs = np.load(post_vec_path)
else:
    post_vecs = spacy_embed_texts(candidate_posts["text"].tolist(), batch_size=64)
    np.save(post_vec_path, post_vecs)

post_norm = np.linalg.norm(post_vecs, axis=1, keepdims=True) + 1e-12
post_vecs_n = post_vecs / post_norm

In [20]:
def user_history_embedding(con, users: list[str], max_comments_per_user: int = 50) -> dict[str, np.ndarray]:
    if not users:
        return {}
    user_list = "(" + ",".join("'" + u.replace("'", "''") + "'" for u in users) + ")"
    df = con.execute(f"""
    WITH sampled AS (
      SELECT
        author,
        body,
        ROW_NUMBER() OVER (PARTITION BY author ORDER BY created_utc DESC) AS rn
      FROM comments_clean
      WHERE author IN {user_list}
        AND body IS NOT NULL
        AND created_utc IS NOT NULL
    )
    SELECT author, body
    FROM sampled
    WHERE rn <= {int(max_comments_per_user)}
    """).df()

    df["body"] = df["body"].fillna("").astype(str)
    emb = {}
    for u, g in df.groupby("author"):
        vecs = spacy_embed_texts(g["body"].tolist(), batch_size=64)
        v = vecs.mean(axis=0)
        n = np.linalg.norm(v) + 1e-12
        emb[u] = v / n
    return emb

In [21]:
def build_user_holdouts(
    con,
    users: list[str],
    candidate_posts: pd.DataFrame,
    post_vecs_n: np.ndarray,
    holdout_frac: float,
    neg_per_pos: int,
    dissimilar_q: float,
    max_comments_for_history: int = 50
) -> pd.DataFrame:

    emb = user_history_embedding(con, users, max_comments_for_history)

    sub_ids = candidate_posts["submission_id"].astype(str).tolist()
    sub_list = "(" + ",".join("'" + s.replace("'", "''") + "'" for s in sub_ids) + ")"
    user_list = "(" + ",".join("'" + u.replace("'", "''") + "'" for u in users) + ")"

    user_pos = con.execute(f"""
    SELECT DISTINCT author, link_submission_id AS submission_id
    FROM comments_clean
    WHERE author IN {user_list}
      AND link_submission_id IN {sub_list}
    """).df()

    pos_map = {u: set() for u in users}
    for r in user_pos.itertuples(index=False):
        if r.submission_id is not None:
            pos_map[r.author].add(str(r.submission_id))

    rows = []
    sub_idx = {sid: i for i, sid in enumerate(sub_ids)}

    for u in users:
        if u not in emb:
            continue
        pos = list(pos_map.get(u, set()))
        if len(pos) < 2:
            continue

        n_pos_hold = max(1, int(math.ceil(len(pos) * holdout_frac)))
        uvec = emb[u]

        sims_pos = []
        for sid in pos:
            i = sub_idx.get(sid)
            if i is None:
                continue
            sims_pos.append((sid, float(np.dot(post_vecs_n[i], uvec))))
        if len(sims_pos) < 2:
            continue

        sims_only = np.array([s for _, s in sims_pos])
        thr = float(np.quantile(sims_only, dissimilar_q))
        dissimilar_pos = [sid for sid, s in sims_pos if s <= thr]
        if len(dissimilar_pos) == 0:
            dissimilar_pos = [sid for sid, _ in sorted(sims_pos, key=lambda x: x[1])[:max(1, n_pos_hold)]]

        pos_hold = random.sample(dissimilar_pos, k=min(n_pos_hold, len(dissimilar_pos)))

        neg_pool = [sid for sid in sub_ids if sid not in pos_map.get(u, set())]
        if len(neg_pool) == 0:
            continue

        sims_neg = []
        pick = min(len(neg_pool), 400)
        for sid in random.sample(neg_pool, k=pick):
            i = sub_idx.get(sid)
            if i is None:
                continue
            sims_neg.append((sid, float(np.dot(post_vecs_n[i], uvec))))

        sims_neg = sorted(sims_neg, key=lambda x: x[1])
        need_neg = min(len(sims_neg), len(pos_hold) * neg_per_pos)
        neg_hold = [sid for sid, _ in sims_neg[:need_neg]]

        for sid in pos_hold:
            i = sub_idx[sid]
            sim = float(np.dot(post_vecs_n[i], uvec))
            rows.append((u, sid, 1, sim, 1))

        for sid in neg_hold:
            i = sub_idx[sid]
            sim = float(np.dot(post_vecs_n[i], uvec))
            rows.append((u, sid, 0, sim, 1))

    out = pd.DataFrame(rows, columns=["author","submission_id","label","user_post_sim","dissimilar_mode"])
    return out

In [24]:
def cap_users(df, n):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=RANDOM_SEED)


# ------------------------------------------------------------
# 1. Fix user set once across all sweeps
# ------------------------------------------------------------
print("Collecting user universe...")
all_users = set()
for samp in samples.values():
    capped = cap_users(samp[["author"]], CONFIG["max_users_for_holdout_eval"])
    all_users.update(capped["author"].tolist())
all_users = sorted(list(all_users))
print(f"Total users for holdout eval: {len(all_users)}")


# ------------------------------------------------------------
# 2. Precompute user history embeddings (PRINT PROGRESS)
# ------------------------------------------------------------
print("Embedding user histories...")
user_emb = {}
BATCH = 200
MAX_COMMENTS = 25   # ↓ from 50, huge speedup, still valid

for i in range(0, len(all_users), BATCH):
    batch = all_users[i:i+BATCH]
    emb = user_history_embedding(
        con,
        users=batch,
        max_comments_per_user=MAX_COMMENTS
    )
    user_emb.update(emb)
    print(f"  embedded {min(i+BATCH, len(all_users))}/{len(all_users)} users")

print(f"User embeddings ready: {len(user_emb)}")


# ------------------------------------------------------------
# 3. Precompute positive interactions
# ------------------------------------------------------------
print("Loading positive interactions...")
sub_ids = candidate_posts["submission_id"].astype(str).tolist()
sub_idx = {sid: i for i, sid in enumerate(sub_ids)}

sub_list = "(" + ",".join("'" + s.replace("'", "''") + "'" for s in sub_ids) + ")"
user_list = "(" + ",".join("'" + u.replace("'", "''") + "'" for u in all_users) + ")"

user_pos_df = con.execute(f"""
SELECT DISTINCT author, link_submission_id AS submission_id
FROM comments_clean
WHERE author IN {user_list}
  AND link_submission_id IN {sub_list}
""").df()

pos_map = {}
for r in user_pos_df.itertuples(index=False):
    if r.submission_id is None:
        continue
    pos_map.setdefault(r.author, set()).add(str(r.submission_id))

print(f"Users with >=1 positive: {len(pos_map)}")


# ------------------------------------------------------------
# 4. Fixed negative pools
# ------------------------------------------------------------
print("Preparing negative pools...")
NEG_POOL_SIZE = 300
rng = np.random.default_rng(RANDOM_SEED)

neg_pool = {}
for u in pos_map.keys():
    positives = pos_map[u]
    pool = [sid for sid in sub_ids if sid not in positives]
    if len(pool) > NEG_POOL_SIZE:
        pool = rng.choice(pool, size=NEG_POOL_SIZE, replace=False).tolist()
    neg_pool[u] = pool

print("Negative pools ready")


# ------------------------------------------------------------
# 5. FAST holdout builder with progress prints
# ------------------------------------------------------------
def fast_holdouts(users, frac, sample_name):
    rows = []
    total = len(users)
    step = max(1, total // 10)

    for idx, u in enumerate(users):
        if idx % step == 0:
            print(f"[{sample_name} | frac={frac}] user {idx}/{total}")

        if u not in user_emb:
            continue
        positives = list(pos_map.get(u, []))
        if len(positives) < 3:
            continue

        uvec = user_emb[u]

        pos_idx = [sub_idx[sid] for sid in positives if sid in sub_idx]
        pos_sims = post_vecs_n[pos_idx] @ uvec
        if len(pos_sims) < 3:
            continue

        n_pos = max(1, int(len(pos_sims) * frac))
        thr = np.quantile(pos_sims, CONFIG["dissimilar_quantile"])

        dissim = [positives[i] for i, s in enumerate(pos_sims) if s <= thr]
        if not dissim:
            dissim = [positives[i] for i in np.argsort(pos_sims)[:n_pos]]

        pos_hold = rng.choice(dissim, size=min(n_pos, len(dissim)), replace=False)

        neg_candidates = neg_pool[u]
        neg_idx = [sub_idx[sid] for sid in neg_candidates if sid in sub_idx]
        neg_sims = post_vecs_n[neg_idx] @ uvec

        order = np.argsort(neg_sims)
        need_neg = min(len(order), len(pos_hold) * CONFIG["negatives_per_positive"])
        neg_hold = [neg_candidates[i] for i in order[:need_neg]]

        for sid in pos_hold:
            rows.append((u, sid, 1, float(post_vecs_n[sub_idx[sid]] @ uvec), 1))
        for sid in neg_hold:
            rows.append((u, sid, 0, float(post_vecs_n[sub_idx[sid]] @ uvec), 1))

    return pd.DataFrame(
        rows,
        columns=["author", "submission_id", "label", "user_post_sim", "dissimilar_mode"]
    )


# ------------------------------------------------------------
# 6. Run sweeps with visibility
# ------------------------------------------------------------
all_holdouts = []

for sample_name, samp in samples.items():
    users = cap_users(
        samp[["author"]],
        CONFIG["max_users_for_holdout_eval"]
    )["author"].tolist()

    print(f"\n=== SAMPLE REGIME: {sample_name} ({len(users)} users) ===")

    for frac in CONFIG["holdout_sweep"]:
        print(f"--- Holdout fraction {frac} ---")
        hdf = fast_holdouts(users, frac, sample_name)
        hdf["sample_regime"] = sample_name
        hdf["holdout_frac"] = frac
        all_holdouts.append(hdf)

holdouts = pd.concat(all_holdouts, axis=0, ignore_index=True)
out_path = os.path.join(CONFIG["outputs_dir"], "holdouts_tasks.parquet")
holdouts.to_parquet(out_path, index=False)

print("\nHoldout task counts:")
print(
    holdouts.groupby(["sample_regime", "holdout_frac", "label"])
    .size()
    .reset_index(name="n")
    .head(20)
)


Total users for holdout eval: 15059
Embedding user histories...
  embedded 200/15059 users
  embedded 400/15059 users
  embedded 600/15059 users
  embedded 800/15059 users
  embedded 1000/15059 users
  embedded 1200/15059 users
  embedded 1400/15059 users
  embedded 1600/15059 users
  embedded 1800/15059 users
  embedded 2000/15059 users
  embedded 2200/15059 users
  embedded 2400/15059 users
  embedded 2600/15059 users
  embedded 2800/15059 users
  embedded 3000/15059 users
  embedded 3200/15059 users
  embedded 3400/15059 users
  embedded 3600/15059 users
  embedded 3800/15059 users
  embedded 4000/15059 users
  embedded 4200/15059 users
  embedded 4400/15059 users
  embedded 4600/15059 users
  embedded 4800/15059 users
  embedded 5000/15059 users
  embedded 5200/15059 users
  embedded 5400/15059 users
  embedded 5600/15059 users
  embedded 5800/15059 users
  embedded 6000/15059 users
  embedded 6200/15059 users
  embedded 6400/15059 users
  embedded 6600/15059 users
  embedded 6800/

In [25]:
print("Saving sampled profile populations...")

for name, samp in samples.items():
    out = samp.copy()
    out_path = os.path.join(CONFIG["outputs_dir"], f"profiles_{name}.parquet")
    out.to_parquet(out_path, index=False)
    print(f"  wrote {name}: {len(out):,} users → {out_path}")

print("Done.")


Saving sampled profile populations...
  wrote threshold_18: 13,350 users → outputs_infer_v3/profiles_threshold_18.parquet
  wrote threshold_30: 6,278 users → outputs_infer_v3/profiles_threshold_30.parquet
  wrote threshold_50: 2,673 users → outputs_infer_v3/profiles_threshold_50.parquet
  wrote stratified: 12,424 users → outputs_infer_v3/profiles_stratified.parquet
  wrote distmatch: 13,350 users → outputs_infer_v3/profiles_distmatch.parquet
  wrote influence_weighted: 13,350 users → outputs_infer_v3/profiles_influence_weighted.parquet
Done.


In [26]:
summary_rows = []

for name, samp in samples.items():
    summary_rows.append({
        "sample_regime": name,
        "n_users": len(samp),
        "median_comments": float(samp["n_comments"].median()),
        "median_comments_per_day": float(samp["comments_per_day"].median()),
        "median_rigidity": float(samp["rigidity"].median()),
        "median_influence": float(samp["influence_basic"].median()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("sample_regime")
summary_df


,sample_regime,n_users,median_comments,median_comments_per_day,median_rigidity,median_influence
4,distmatch,13350,3.0,0.043893,0.429808,1.791759
5,influence_weighted,13350,2.0,1.000000,0.707100,1.098612
3,stratified,12424,2.0,1.000000,0.707100,1.098612
0,threshold_18,13350,28.0,0.051510,0.161052,4.651733
1,threshold_30,6278,45.0,0.069621,0.140787,5.302822
2,threshold_50,2673,70.0,0.096037,0.132187,5.972769


In [27]:
manifest = {
    "duckdb_path": CONFIG["duckdb_path"],
    "outputs_dir": CONFIG["outputs_dir"],

    "profiles_files": {
        name: f"profiles_{name}.parquet"
        for name in samples.keys()
    },

    "holdouts_tasks_file": "holdouts_tasks.parquet",

    "nela": {
        "db_path": CONFIG["nela_db_path"],
        "labels_csv": CONFIG["nela_labels_csv"],
        "items_file": "nela_items.parquet"
    },

    "holdout_config": {
        "candidate_posts_topn": CONFIG["candidate_posts_topn"],
        "holdout_sweep": CONFIG["holdout_sweep"],
        "negatives_per_positive": CONFIG["negatives_per_positive"],
        "dissimilar_quantile": CONFIG["dissimilar_quantile"],
        "max_users_for_eval": CONFIG["max_users_for_holdout_eval"]
    },

    "random_seed": RANDOM_SEED
}

manifest_path = os.path.join(CONFIG["outputs_dir"], "infer_v3_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest written →", manifest_path)


Manifest written → outputs_infer_v3/infer_v3_manifest.json


In [28]:
try:
    con.close()
    print("DuckDB connection closed.")
except Exception:
    pass


DuckDB connection closed.
